## Operational Flow 
__In the notebook we will design the complete operation flow from simple text SQL query to reasoning on related AST instance graph and reconstructing updated SQL text query back based on the reasoning results.__



In [1]:
query = "SELECT account_id, SUM(amount) AS total FROM transactions WHERE account_id = 123 GROUP BY account_id"
query = "SELECT account_id, account_to, bank_to, amount FROM Order;"

### Step-1 : SQL to AST conversion

In [2]:
from sqlglot import parse_one, exp

In [3]:
ast_obj = parse_one(query)

### Step-2 : AST (`sqlgpt_parser`) to AST Tree conversion

In [4]:
import uuid

In [5]:
# --- Type Mapper ---

CLAUSE_TYPES = {
    "From": "FromClause",
    "Group": "GroupByClause",
    "Into": "IntoClause",
    "Limit": "LimitClause",
    "Select": "SelectClause",
    "Set": "SetClause",
    "Update": "UpdateClause",
    "Values": "ValuesClause",
    "Where": "WhereClause"
}

STATEMENT_TYPES = {
    "Delete": "DeleteStatement",
    "Insert": "InsertStatement",
    "Update": "UpdateStatement",
    "Select": "SelectStatement"
}

EXPRESSION_CATEGORIES = {
    "Alias": "Alias",
    "Table": "TableRef",
    "Column": "ColumnRef",
    "Wildcard": "WildCard",
    "Identifier": "Identifier",
    # Functions
    "Sum": "Function",
    "Count": "Function",
    "Avg": "Function",
    "Max": "Function",
    "Min": "Function",
    "Concat": "Function",
    "Coalesce": "Function",
    # Operators
    "And": "Operator",
    "Or": "Operator",
    "EQ": "Operator",
    "GT": "Operator",
    "LT": "Operator",
    "Add": "Operator",
    "Sub": "Operator",
    "Mul": "Operator",
    "Div": "Operator",
    # Literals
    "Literal": "Literal"
}

def map_node_type(node):
    class_name = type(node).__name__
    if class_name in STATEMENT_TYPES:
        return "Statement", STATEMENT_TYPES[class_name]
    elif class_name in CLAUSE_TYPES:
        return "Clause", CLAUSE_TYPES[class_name]
    elif class_name in EXPRESSION_CATEGORIES:
        return "Expression", EXPRESSION_CATEGORIES[class_name]
    else:
        return "Expression", class_name


In [6]:

# --- TreeNode Class ---

class TreeNode:
    def __init__(self, sqlglot_node, parent=None):
        self.remove = False
        self.parent = parent
        self.children = []
        self.sqlglot_node = sqlglot_node
        self.kind, self.name = map_node_type(sqlglot_node)

        self.id = "n"+str(uuid.uuid4())[:6]
        sqlglot_node.id = self.id

        if self.name == "ColumnRef":
            self.refcol = None
            self.reftable = None
            # self.reference_id = None
            parts = sqlglot_node.parts
            if len(parts) == 2:
                self.reftable = parts[0].this
                self.refcol = parts[1].this
            elif len(parts) == 1:
                self.refcol = parts[0].this

        if self.name == "TableRef":
            self.reftable = sqlglot_node.name
            # self.reference_id = None

        if self.name == "Alias":
            alias_expr = sqlglot_node.args.get("alias")
            if isinstance(alias_expr, exp.Identifier):
                self.alias = alias_expr.name

        if self.name == "Literal":
            self.value = sqlglot_node.this

    def add_child(self, child):
        self.children.append(child)

    def __repr__(self):
        suffix = f" ({self.name})"
        if self.name == "ColumnRef":
            suffix += f" [reftable={self.reftable}, refcol={self.refcol}]"
        if self.name == "TableRef":
            suffix += f" [reftable={self.reftable}]"
        if self.name == "Alias" and hasattr(self, "alias"):
            suffix += f" [name={self.alias}]"
        if self.name == "Literal" and hasattr(self, "value"):
            suffix += f" [value={self.value}]"

        if hasattr(self, "reference_id"):
            suffix += f" [reference_id={self.reference_id}]"
        if hasattr(self, "remove"):
            suffix += f" [remove={self.remove}]"
        return f"<({self.id}) {self.kind} | {suffix}>"

    def walk(self):
        yield self
        for child in self.children:
            yield from child.walk()

    def print_tree(self, level=0):
        print("  " * level + repr(self))
        for child in self.children:
            child.print_tree(level + 1)


In [7]:

# --- Tree Builder ---


def build_tree(sqlglot_node):
    kind, name = map_node_type(sqlglot_node)
    stmt_node = TreeNode(sqlglot_node)
    stmt_node.kind = "Statement"
    stmt_node.name = name
    stmt_node.id[0].replace("n", "q")  # Change the first character to 's' for statement nodes

    clause_root = TreeNode(sqlglot_node, parent=stmt_node)
    clause_root.kind = "Clause"
    clause_root.id[0].replace("n", "c")  # Change the first character to 'c' for clause nodes
    clause_root.name = CLAUSE_TYPES.get(type(sqlglot_node).__name__, type(sqlglot_node).__name__ + "Clause")
    stmt_node.add_child(clause_root)

    found_first_clause = False
    for key, child in sqlglot_node.args.items():
        if child is None:
            continue

        children = child if isinstance(child, list) else [child]

        for expr in children:
            if not isinstance(expr, exp.Expression):
                continue

            c_kind, c_name = map_node_type(expr)

            if not found_first_clause and (key != "from" and c_kind != "Clause"):
                sub = _build_recursive(expr, clause_root)
                if sub:
                    clause_root.add_child(sub)
            else:
                found_first_clause = True
                clause_node = TreeNode(expr, parent=stmt_node)
                clause_node.kind = "Clause"
                clause_node.name = c_name
                stmt_node.add_child(clause_node)

                for _, grandchild in expr.args.items():
                    if isinstance(grandchild, exp.Expression):
                        if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                            continue  # Skip Identifier under Table
                        child_node = _build_recursive(grandchild, clause_node)
                        if child_node:
                            clause_node.add_child(child_node)
                    elif isinstance(grandchild, list):
                        for item in grandchild:
                            if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                continue  # Skip Identifier under Table
                            if isinstance(item, exp.Expression):
                                child_node = _build_recursive(item, clause_node)
                                if child_node:
                                    clause_node.add_child(child_node)

    return stmt_node


def _build_recursive(sqlglot_node, parent_node):
    if isinstance(sqlglot_node, exp.Identifier) and isinstance(parent_node, TreeNode) and parent_node.name == "ColumnRef":
        return None

    if isinstance(parent_node, TreeNode) and parent_node.name == "TableRef" and isinstance(sqlglot_node, exp.Identifier):
        return None

    current = TreeNode(sqlglot_node, parent_node)

    if isinstance(sqlglot_node, exp.Select):
        clause_root = TreeNode(sqlglot_node, parent=current)
        clause_root.kind = "Clause"
        clause_root.name = "SelectClause"
        current.add_child(clause_root)

        found_first_clause = False
        for key, child in sqlglot_node.args.items():
            if child is None:
                continue

            children = child if isinstance(child, list) else [child]
            for expr in children:
                if not isinstance(expr, exp.Expression):
                    continue
                c_kind, c_name = map_node_type(expr)

                if not found_first_clause and (key != "from" and c_kind != "Clause"):
                    sub = _build_recursive(expr, clause_root)
                    if sub:
                        clause_root.add_child(sub)
                else:
                    found_first_clause = True
                    clause_node = TreeNode(expr, parent=current)
                    clause_node.kind = "Clause"
                    clause_node.name = c_name
                    current.add_child(clause_node)

                    for _, grandchild in expr.args.items():
                        if isinstance(grandchild, exp.Expression):
                            if isinstance(expr, exp.Table) and isinstance(grandchild, exp.Identifier):
                                continue
                            child_node = _build_recursive(grandchild, clause_node)
                            if child_node:
                                clause_node.add_child(child_node)
                        elif isinstance(grandchild, list):
                            for item in grandchild:
                                if isinstance(expr, exp.Table) and isinstance(item, exp.Identifier):
                                    continue
                                if isinstance(item, exp.Expression):
                                    child_node = _build_recursive(item, clause_node)
                                    if child_node:
                                        clause_node.add_child(child_node)
    else:
        for _, child in sqlglot_node.args.items():
            if isinstance(child, exp.Expression):
                if isinstance(sqlglot_node, exp.Table) and isinstance(child, exp.Identifier):
                    continue
                child_node = _build_recursive(child, current)
                if child_node:
                    current.add_child(child_node)
            elif isinstance(child, list):
                for item in child:
                    if isinstance(sqlglot_node, exp.Table) and isinstance(item, exp.Identifier):
                        continue
                    if isinstance(item, exp.Expression):
                        child_node = _build_recursive(item, current)
                        if child_node:
                            current.add_child(child_node)

    return current


In [8]:
ast = parse_one(query)
tree_root = build_tree(ast)

# Print the tree
tree_root.print_tree()

<(n64266d) Statement |  (SelectStatement) [remove=False]>
  <(nd44068) Clause |  (SelectClause) [remove=False]>
    <(ne11b3f) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [remove=False]>
    <(n607b45) Expression |  (ColumnRef) [reftable=None, refcol=account_to] [remove=False]>
    <(ned6ebf) Expression |  (ColumnRef) [reftable=None, refcol=bank_to] [remove=False]>
    <(n22eae4) Expression |  (ColumnRef) [reftable=None, refcol=amount] [remove=False]>
  <(nce6d40) Clause |  (FromClause) [remove=False]>
    <(n9233bb) Expression |  (TableRef) [reftable=Order] [remove=False]>


### Step-3 : AST Tree to Ontology Instance 

In [9]:
# load ontology
from owlready2 import get_ontology, sync_reasoner
import uuid

onto_path = "../ontology_file/aput.rdf"
onto = get_ontology(onto_path).load()

In [10]:
def resolve_references_with_ontology(root_node, ontology):
    # table_entities = {
    #     t.TableName[0]: t.name
    #     for t in onto.individuals()
    #         if "Table" == t.is_a[0].name 
    # }
    table_entities = {}
    for t in onto.individuals():
        if len(t.is_a) == 2 and "Table" == t.is_a[1].name:
            table_entities[t.TableName[0]] = t.name
    print(table_entities)
    
    column_lookup = {}
    for n in onto.individuals():
        if "Column" == n.is_a[0].name:
            for tbl in getattr(n, "columnOfTable", []):
                table_name = tbl.TableName[0]
                column_lookup[(table_name, n.ColumnName[0])] = n.name
    print(column_lookup)

    def resolve_for_statement(stmt_node):
        alias_map = {}  # alias -> table name
        tables_in_scope = []  # actual tables in FROM clause

        for child in stmt_node.children:
            if child.name == "FromClause":
                for t in child.walk():
                    if t.name == "TableRef" and hasattr(t, "reftable"):
                        table_name = t.reftable
                        alias_map[table_name] = table_name
                        tables_in_scope.append(table_name)

                        for sub in t.children:
                            if sub.name == "Alias" and hasattr(sub, "alias"):
                                alias_map[sub.alias] = table_name

        for node in stmt_node.walk():
            if node.name == "TableRef" and hasattr(node, "reftable"):
                tbl_key = node.reftable
                ref_table = table_entities.get(tbl_key)
                if ref_table:
                    node.reference_id = table_entities[node.reftable]

            elif node.name == "ColumnRef" and hasattr(node, "refcol"):
                # Use declared reftable if exists
                if hasattr(node, "reftable") and node.reftable:
                    tbl_key = alias_map.get(node.reftable, node.reftable)
                    col_key = (tbl_key, node.refcol)
                else:
                    if len(tables_in_scope) == 1:
                        tbl_key = tables_in_scope[0]
                        print(f"[Info] Using single table '{tbl_key}' for column '{node.refcol}'.")
                        col_key = (tbl_key, node.refcol)
                    else:
                        print(f"[Warning] Ambiguous column '{node.refcol}' with no reftable in multi-table query.")
                        continue
                

                ref_col = column_lookup.get(col_key)
                print(f"[Debug] Resolving ColumnRef {ref_col} with key {col_key}.")
                if ref_col:
                    node.reference_id = ref_col

    for node in root_node.walk():
        if node.kind == "Statement":
            resolve_for_statement(node)


In [11]:
resolve_references_with_ontology(tree_root, onto)
tree_root.print_tree()

{'Order': 't001', 'Transaction': 't002', 'Account': 't003', 'Loan': 't004', 'Disposition': 't005', 'Client': 't006', 'District': 't008', 'Card': 't007'}
{('Order', 'order_id'): 'c001', ('Order', 'account_id'): 'c002', ('Transaction', 'account_id'): 'c002', ('Account', 'account_id'): 'c002', ('Loan', 'account_id'): 'c002', ('Disposition', 'account_id'): 'c002', ('Order', 'bank_to'): 'c003', ('Order', 'account_to'): 'c004', ('Order', 'amount'): 'c005', ('Order', 'k_symbol'): 'c006', ('Transaction', 'trans_id'): 'c007', ('Transaction', 'date'): 'c008', ('Transaction', 'type'): 'c009', ('Transaction', 'operation'): 'c010', ('Transaction', 'balance'): 'c011', ('Transaction', 'bank'): 'c012', ('Transaction', 'account'): 'c013', ('Account', 'district_id'): 'c014', ('Client', 'district_id'): 'c014', ('District', 'district_id'): 'c014', ('Account', 'frequency'): 'c015', ('Loan', 'loan_id'): 'c016', ('Loan', 'duration'): 'c017', ('Loan', 'payments'): 'c018', ('Loan', 'status'): 'c019', ('Disposi

### Ontology reasoning

In [12]:
with onto:

    resolve_references_with_ontology(tree_root, onto)
    
    class_lookup = {cls.name: cls for cls in onto.classes()}

    # ✅ Tracking lists
    table_ref_instances = []
    column_ref_instances = []

    def walk_and_instantiate(node, parent_instance=None):
        node_class = class_lookup.get(node.name)
        if not node_class:
            print(f"[Warning] Missing class in ontology: {node.name}")
            return None

        inst = node_class(node.id)

        # ✅ Attach parent link
        if parent_instance:
            inst.immediateParentNode.append(parent_instance)

        if node.name == "SelectStatement":
            ref_obj = onto.search_one(iri="*a0012")
            inst.executedBy.append(ref_obj)

        # ✅ Link TableRef to ontology Table via reference_id
        if node.name == "TableRef":
            table_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToTable.append(ref_obj)
                else:
                    print(f"[Warning] TableRef unresolved: {node.reference_id}")

        # ✅ Link ColumnRef to ontology Column via reference_id
        if node.name == "ColumnRef":
            column_ref_instances.append(inst)
            if hasattr(node, "reference_id"):
                ref_obj = onto.search_one(iri="*" + node.reference_id)
                if ref_obj:
                    inst.referencesToColumn.append(ref_obj)
                else:
                    print(f"[Warning] ColumnRef unresolved: {node.reference_id}")

        # ✅ Recurse
        for child in node.children:
            walk_and_instantiate(child, inst)

        return inst

    # 🔄 Begin instantiation from root
    walk_and_instantiate(tree_root)

    sync_reasoner(infer_property_values=True, debug=0)

    onto.save(file="../ontology_file/resolved_query_instances.owl", format="rdfxml")


{'Order': 't001', 'Transaction': 't002', 'Account': 't003', 'Loan': 't004', 'Disposition': 't005', 'Client': 't006', 'District': 't008', 'Card': 't007'}
{('Order', 'order_id'): 'c001', ('Order', 'account_id'): 'c002', ('Transaction', 'account_id'): 'c002', ('Account', 'account_id'): 'c002', ('Loan', 'account_id'): 'c002', ('Disposition', 'account_id'): 'c002', ('Order', 'bank_to'): 'c003', ('Order', 'account_to'): 'c004', ('Order', 'amount'): 'c005', ('Order', 'k_symbol'): 'c006', ('Transaction', 'trans_id'): 'c007', ('Transaction', 'date'): 'c008', ('Transaction', 'type'): 'c009', ('Transaction', 'operation'): 'c010', ('Transaction', 'balance'): 'c011', ('Transaction', 'bank'): 'c012', ('Transaction', 'account'): 'c013', ('Account', 'district_id'): 'c014', ('Client', 'district_id'): 'c014', ('District', 'district_id'): 'c014', ('Account', 'frequency'): 'c015', ('Loan', 'loan_id'): 'c016', ('Loan', 'duration'): 'c017', ('Loan', 'payments'): 'c018', ('Loan', 'status'): 'c019', ('Disposi

In [13]:
for inst in table_ref_instances + column_ref_instances:
    print(f"{inst.name} → Alignment: {getattr(inst, 'hasAlignmentState', 'N/A')}")


n9233bb → Alignment: []
ne11b3f → Alignment: []
n607b45 → Alignment: []
ned6ebf → Alignment: []
n22eae4 → Alignment: [aput.Violated]


### Ontology to AST

In [18]:
column_ref_dict = {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in column_ref_instances }

table_ref_dict = {inst.name: "Aligned" if getattr(inst, 'hasAlignmentState', []) == [] else "Violated" for inst in table_ref_instances}

In [17]:
column_ref_dict

{'ne11b3f': 'Aligned',
 'n607b45': 'Aligned',
 'ned6ebf': 'Aligned',
 'n22eae4': 'Violated'}

In [25]:
def soft_prune_ast(root, column_ref_instances, table_ref_instances):
    """
    Traverse TreeNode tree and soft-prune by setting `remove = True` on nodes based on alignment states.
    """

    for node in root.walk():
        # Handle ColumnRef node logic
        if node.name == "ColumnRef":
            state = column_ref_instances.get(node.id)
            if state == "Violated":
                print(f"ColumnRef node {node.id} has alignment state Violated.")
                parent = node.parent
                print(f"Parent of ColumnRef node {node.id} is {parent.name if parent else 'None'}.")
                if parent:
                    if parent.name == "SelectClause":
                        node.remove = True
                        print(f"Marking ColumnRef node {node.id} for removal (in Clause).")
                    elif parent.name in ("Operator", "Function"):
                        parent.remove = True
                        print(f"Marking parent {parent.id} of ColumnRef node {node.id} for removal (in Operator/Function).")

        # Handle TableRef node logic
        elif node.name == "TableRef":
            state= table_ref_instances.get(node.id)
            if state == "Violated":
                # Find immediate statement node
                parent = node.parent
                while parent and parent.kind != "Statement":
                    parent = parent.parent
                if parent:
                    parent.remove = True
                    print(f"Marking Statement node {parent.id} for removal due to TableRef {node.id}.")


In [26]:
column_ref_instances

[aput.ne11b3f, aput.n607b45, aput.ned6ebf, aput.n22eae4]

In [27]:
soft_prune_ast(tree_root, column_ref_dict, table_ref_dict)

# To inspect:
tree_root.print_tree()



ColumnRef node n22eae4 has alignment state Violated.
Parent of ColumnRef node n22eae4 is SelectClause.
Marking ColumnRef node n22eae4 for removal (in Clause).
<(n64266d) Statement |  (SelectStatement) [remove=False]>
  <(nd44068) Clause |  (SelectClause) [remove=False]>
    <(ne11b3f) Expression |  (ColumnRef) [reftable=None, refcol=account_id] [reference_id=c002] [remove=False]>
    <(n607b45) Expression |  (ColumnRef) [reftable=None, refcol=account_to] [reference_id=c004] [remove=False]>
    <(ned6ebf) Expression |  (ColumnRef) [reftable=None, refcol=bank_to] [reference_id=c003] [remove=False]>
    <(n22eae4) Expression |  (ColumnRef) [reftable=None, refcol=amount] [reference_id=c005] [remove=True]>
  <(nce6d40) Clause |  (FromClause) [remove=False]>
    <(n9233bb) Expression |  (TableRef) [reftable=Order] [reference_id=t001] [remove=False]>


### AST to sql

In [32]:
def remove_column(ast, node_id):
    """
    Remove a column (or expression) from a SELECT AST node by matching its id.
    """
    select_expressions = ast.expressions
    ast.set(
        "expressions",
        [expr for expr in select_expressions if not (hasattr(expr, 'id') and expr.id == node_id)]
    )


In [33]:
def apply_pruning_to_ast(tree_root, ast):
    """
    Traverse TreeNode tree and remove corresponding sqlglot AST nodes for nodes marked `.remove = True`.
    """
    for node in tree_root.walk():
        if getattr(node, "remove", False):
            # Remove corresponding sqlglot node from AST if it's a ColumnRef under SELECT
            if node.name == "ColumnRef":
                remove_column(ast, node.id)


In [34]:
apply_pruning_to_ast(tree_root, ast)

In [35]:
ast.sql()

'SELECT account_id, account_to, bank_to FROM Order'